# RNN Text Classification (LSTM vs GRU) - نسخة المحاضر

هذا الدفتر مخصص للمحاضر، مع شرح تفصيلي لكل خطوة: ماذا نفعل، ولماذا، وكيف نفسر النتائج للطلاب.

## الهدف التعليمي
تصنيف مشاعر (sentiment) من نصوص مراجعات: إيجابي (1) أو سلبي (0).
مقارنة **LSTM** و **GRU**.

## خطة الشرح
1. استيراد المكتبات
2. قراءة البيانات
3. Tokenization + padding
4. تقسيم البيانات
5. بناء LSTM
6. تدريب LSTM
7. بناء GRU
8. تدريب GRU
9. مقارنة + تنبؤ على جمل


## الخطوة 1: استيراد المكتبات

Tokenizer يحوّل النص لأرقام؛ pad_sequences يوحّد الطول.


In [ ]:
# الخطوة 1) استيراد المكتبات
# pip install tensorflow -q
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Dense, Dropout


## الخطوة 2: قراءة البيانات

482 مراجعة متوازنة (241 إيجابي + 241 سلبي).


In [ ]:
# Step 2) قراءة البيانات / Load dataset
import os
import urllib.request

filename = 'sentiment_reviews.csv'
if not os.path.exists(filename):
    url = 'https://raw.githubusercontent.com/iksasa15/AI-ML/main/code/15-%20Deep%20Learning/5-%20RNN%20Text%20Classification/sentiment_reviews.csv'
    urllib.request.urlretrieve(url, filename)

dataset = pd.read_csv(filename)
dataset.head()


## الخطوة 3: Tokenization + Padding

- **MAX_WORDS=2000**: أقصى حجم للمفردات
- **MAX_LEN=20**: طول كل sequence (قص أو padding)
- **oov_token**: كلمات غير معروفة → `<OOV>`

النص → قائمة أرقام → مصفوفة (n_samples, 20)


In [ ]:
# الخطوة 3) Tokenization + padding
MAX_WORDS = 2000
MAX_LEN = 20

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(dataset['review'])
sequences = tokenizer.texts_to_sequences(dataset['review'])
X = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')
y = dataset['sentiment'].values
print('X shape:', X.shape)


## الخطوة 4: تقسيم البيانات

`stratify=y` للحفاظ على توازن الفئات.


In [ ]:
# الخطوة 4) تقسيم البيانات
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)


## الخطوة 5: بناء LSTM

البنية: Embedding → LSTM(64) → Dropout → Dense → sigmoid

- **Embedding**: كل كلمة → متجه كثيف 64 بعد
- **LSTM**: ذاكرة طويلة المدى — يلتقط ترتيب الكلمات


In [ ]:
# الخطوة 5) بناء LSTM
def build_rnn_model(rnn_layer):
    model = Sequential([
        Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
        rnn_layer,
        Dropout(0.3),
        Dense(32, activation='relu'),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

lstm_model = build_rnn_model(LSTM(64))
lstm_model.summary()


## الخطوة 6: تدريب LSTM

15 epochs مع validation_split=0.2.


In [ ]:
# الخطوة 6) تدريب LSTM
lstm_history = lstm_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
lstm_loss, lstm_acc = lstm_model.evaluate(X_test, y_test, verbose=0)
print(f'LSTM test accuracy: {lstm_acc:.2%}')


## الخطوة 7: بناء GRU

**GRU** أبسط وأسرع من LSTM — بوابتان بدلاً من ثلاث.
مناسب لنصوص قصيرة مثل المراجعات.


In [ ]:
# الخطوة 7) بناء GRU
gru_model = build_rnn_model(GRU(64))
gru_model.summary()


## الخطوة 8: تدريب GRU


In [ ]:
# الخطوة 8) تدريب GRU
gru_history = gru_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=15,
    batch_size=32,
    verbose=1
)
gru_loss, gru_acc = gru_model.evaluate(X_test, y_test, verbose=0)
print(f'GRU test accuracy: {gru_acc:.2%}')


## الخطوة 9: مقارنة + تنبؤ

sentiment: 0 = سلبي، 1 = إيجابي.
جرّب جمل من التدريب واشرح لماذا قد يختلف LSTM عن GRU.


In [ ]:
# الخطوة 9) مقارنة + تنبؤ على جملة جديدة
print('--- مقارنة RNN ---')
print(f'LSTM: {lstm_acc:.2%}')
print(f'GRU:  {gru_acc:.2%}')

sample_reviews = [
    'I loved this product it works perfectly',
    'Terrible quality broke after one day'
]
sample_seq = pad_sequences(tokenizer.texts_to_sequences(sample_reviews), maxlen=MAX_LEN)
lstm_preds = (lstm_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()
gru_preds = (gru_model.predict(sample_seq, verbose=0) > 0.5).astype(int).flatten()

for i, text in enumerate(sample_reviews):
    print(f"\nReview: {text}")
    print(f"LSTM sentiment: {lstm_preds[i]} | GRU sentiment: {gru_preds[i]}")
